# Convert robomimic to LeRobot

This notebook uses the `test_v15.hdf5` sample with two RGB cameras to demonstrate the complete `inspect → plan → convert → validate` workflow. Section 7 provides an optional LeRobot dataset merge example. Run the cells in order.

In [1]:
from pathlib import Path
from pprint import pprint

from leport import convert, create_plan, inspect, merge, validate
from leport.conversion.plan import save_plan
from leport.sources import EpisodeSelection

project_root = Path.cwd().resolve()
if not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("Project root not found; start the notebook from the leport root or notebooks/")

source_path = project_root / "data/robomimic/test/test_v15.hdf5"
output_root = project_root / "outputs"
target_root = output_root / "lerobot-image-demo"
plan_path = output_root / "robomimic-image-demo.yaml"

if not source_path.is_file():
    raise FileNotFoundError(f"Tutorial dataset not found: {source_path}")

print("Project root:", project_root)
print("Source dataset:", source_path)
print("Plan file:", plan_path)
print("Target directory:", target_root)

Project root: /Users/xzhu/Documents/VLA/leport
Source dataset: /Users/xzhu/Documents/VLA/leport/data/robomimic/test/test_v15.hdf5
Plan file: /Users/xzhu/Documents/VLA/leport/outputs/robomimic-image-demo.yaml
Target directory: /Users/xzhu/Documents/VLA/leport/outputs/lerobot-image-demo


## 1. Inspect: read-only source inspection

Inspect all episodes, source fields, dtypes, shapes, and metadata before choosing mappings. This stage does not create a plan or target directory.

In [2]:
inspection = inspect(source_path, adapter="robomimic")

# The complete inspection contains every episode and source field.
# A compact summary keeps the tutorial output readable.
inspection_summary = {
    "adapter": inspection.adapter,
    "episode_count": len(inspection.episode_ids),
    "first_episode_ids": inspection.episode_ids[:5],
    "total_frames": inspection.total_frames,
    "field_count": len(inspection.fields),
    "filter_keys": inspection.metadata.get("filter_keys", []),
    "diagnostics": inspection.diagnostics,
}
pprint(inspection_summary)

{'adapter': 'robomimic',
 'diagnostics': (),
 'episode_count': 10,
 'field_count': 34,
 'filter_keys': ['train', 'valid'],
 'first_episode_ids': ('demo_0', 'demo_1', 'demo_2', 'demo_3', 'demo_4'),
 'total_frames': 531}


In [3]:
# Each row describes a source selector available to ConversionPlan. A directly mapped field must
# have schema_consistent=True and no missing_episodes.
print(f"{'selector':<42} {'dtype':<12} {'shape':<18}")
for field in inspection.fields:
    dtype_text = ", ".join(field.dtypes)
    shape_text = " / ".join(str(shape) for shape in field.shapes)
    print(f"{field.selector:<42} {dtype_text:<12} {shape_text:<18} ")

selector                                   dtype        shape             
actions                                    float64      (7,)               
dones                                      int64        ()                 
next_obs/agentview_depth                   float32      (84, 84, 1)        
next_obs/agentview_image                   uint8        (84, 84, 3)        
next_obs/object                            float64      (10,)              
next_obs/robot0_eef_pos                    float64      (3,)               
next_obs/robot0_eef_quat                   float64      (4,)               
next_obs/robot0_eef_vel_ang                float64      (3,)               
next_obs/robot0_eef_vel_lin                float64      (3,)               
next_obs/robot0_eye_in_hand_depth          float32      (84, 84, 1)        
next_obs/robot0_eye_in_hand_image          uint8        (84, 84, 3)        
next_obs/robot0_gripper_qpos               float64      (2,)               
next_obs/robo

## 2. Define conversion semantics and select episodes

This example converts every episode. It concatenates end-effector pose and gripper position into the numeric state and maps the external and wrist cameras to two LeRobot video features.

In [4]:
# Empty episode_ids with no filter_key selects every episode rather than an empty set.
# Use EpisodeSelection(filter_key="20_percent") to select only that mask.
episode_selection = EpisodeSelection()

# These values describe robot semantics that must be confirmed by the user, not inferred from names or shapes.
fps = 20
task = "lift the cube"
action_source = "actions"
state_sources = (
    "obs/robot0_eef_pos",
    "obs/robot0_eef_quat",
    "obs/robot0_gripper_qpos",
)
# Each key is a robomimic source selector and each value is the complete LeRobot feature name.
# Both image fields are uint8 with single-frame shape (84, 84, 3), which satisfies video output requirements.
image_sources = {
    "obs/agentview_image": "observation.images.agentview",
    "obs/robot0_eye_in_hand_image": "observation.images.wrist",
}

# Actions and the three numeric state fields are float64 in this sample. The plan explicitly casts
# them to the compact float32 representation commonly used for training; images remain uint8.
action_dtype = "float32"
state_dtype = "float32"

## 3. Plan: create and inspect a ConversionPlan

`create_plan` reinspects the selected episodes and builds a strict plan from explicit arguments. Both `image_sources` are provided explicitly, and `use_videos=True` encodes them as MP4.

In [5]:
plan = create_plan(
    source_path,
    target_root=target_root,
    repo_id="local/robomimic-image-demo",
    robot_type="panda",
    fps=fps,
    task=task,
    action_source=action_source,
    action_dtype=action_dtype,
    state_sources=state_sources,
    state_dtype=state_dtype,
    image_sources=image_sources,
    use_videos=True,
    adapter="robomimic",
    selection=episode_selection,
)

# Review fps, task, target feature shapes/dtypes, and source selector order before saving.
pprint(plan.to_dict())

{'adapter': 'robomimic',
 'features': {'action': {'dtype': 'float32', 'shape': [7]},
              'observation.images.agentview': {'dtype': 'video',
                                               'shape': [84, 84, 3]},
              'observation.images.wrist': {'dtype': 'video',
                                           'shape': [84, 84, 3]},
              'observation.state': {'dtype': 'float32', 'shape': [9]}},
 'fps': 20,
 'mappings': {'action': {'cast': 'float32',
                         'operation': 'direct',
                         'sources': ['actions']},
              'observation.images.agentview': {'operation': 'direct',
                                               'sources': ['obs/agentview_image']},
              'observation.images.wrist': {'operation': 'direct',
                                           'sources': ['obs/robot0_eye_in_hand_image']},
              'observation.state': {'cast': 'float32',
                                    'operation': 'concat',
    

## 4. Save the plan

The plan file is the reproducible semantic input to conversion. LePort rejects overwriting an existing plan so reviewed configuration is not lost.

In [6]:
output_root.mkdir(parents=True, exist_ok=True)
if plan_path.exists():
    raise FileExistsError(f"Plan already exists: {plan_path}. Review and move or remove it explicitly.")

saved_plan_path = save_plan(plan, plan_path)
print("Plan saved:", saved_plan_path)

Plan saved: /Users/xzhu/Documents/VLA/leport/outputs/robomimic-image-demo.yaml


## 5. Convert: execute the conversion

Conversion runs preflight before writing to a sibling staging directory. The result is committed atomically only after finalization and LeRobot reload validation succeed.

In [7]:
if target_root.exists():
    raise FileExistsError(
        f"Target already exists: {target_root}. LePort does not overwrite or append datasets."
    )

# Loading the saved YAML verifies that the persisted and in-memory plans
# have the same execution semantics.
conversion_result = convert(plan_path)
pprint(
    {
        "target": str(conversion_result.target),
        "total_episodes": conversion_result.validation.total_episodes,
        "total_frames": conversion_result.validation.total_frames,
        "episode_lengths": conversion_result.validation.episode_lengths,
        "tasks": conversion_result.validation.tasks,
    }
)

/Users/xzhu/Documents/VLA/leport/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Map:   0%|          | 0/59 [00:00<?, ? examples/s]

Map: 100%|██████████| 59/59 [00:00<00:00, 2348.79 examples/s]

[libx264 @ 0x8e943a300] frame I:3     Avg QP:17.38  size:   397
[libx264 @ 0x8e943a300] frame P:19    Avg QP:20.74  size:   141
[libx264 @ 0x8e943a300] frame B:37    Avg QP:23.05  size:    50
[libx264 @ 0x8e943a300] consecutive B-frames: 10.2% 10.2% 25.4% 54.2%
[libx264 @ 0x8e943a300] mb I  I16..4: 44.4% 33.3% 22.2%
[libx264 @ 0x8e943a300] mb P  I16..4: 16.1%  5.8%  0.6%  P16..4: 11.7%  5.7%  1.9%  0.0%  0.0%    skip:58.2%
[libx264 @ 0x8e943a300] mb B  I16..4:  2.9%  0.5%  0.0%  B16..8:  7.7%  5.2%  0.4%  direct: 1.9%  skip:81.5%  L0:37.3% L1:36.9% BI:25.8%
[libx264 @ 0x8e943a300] 8x8 transform intra:26.8% inter:8.1%
[libx264 @ 0x8e943a300] coded y,uvDC,uvAC intra: 20.0% 14.4% 2.9% inter: 4.4% 1.9% 0.9%
[libx264 @ 0x8e943a300] i16 v,h,dc,p: 35% 52% 11%  2%
[libx264 @ 0x8e943a300] i8 v,h,dc,ddl,ddr,vr,hd,vl,hu: 21% 19% 59%  0%  0%  0%  0%  0%  0%
[libx264 @ 0x8e943a300] i4 v,h,dc,ddl,ddr,vr,hd,vl,hu:  9% 47% 29%  1%  3%  0%  3%  2%  5%
[libx264 @ 0x8e943a300] i8c dc,h,v,p: 61% 36%  3%  

Map:   0%|          | 0/58 [00:00<?, ? examples/s]

...

Map: 100%|██████████| 51/51 [00:00<00:00, 4539.39 examples/s]

[libx264 @ 0xa68e31180] frame I:3     Avg QP:16.33  size:   796
[libx264 @ 0xa68e31180] frame P:21    Avg QP:27.68  size:   241
[libx264 @ 0xa68e31180] frame B:27    Avg QP:31.51  size:   106
[libx264 @ 0xa68e31180] consecutive B-frames: 19.6% 23.5% 17.6% 39.2%
[libx264 @ 0xa68e31180] mb I  I16..4: 23.1% 36.1% 40.7%
[libx264 @ 0xa68e31180] mb P  I16..4:  0.0%  0.1%  0.3%  P16..4:  7.1%  7.3%  7.3%  0.0%  0.0%    skip:77.9%
[libx264 @ 0xa68e31180] mb B  I16..4:  0.1%  0.0%  0.0%  B16..8:  4.4%  5.1%  1.4%  direct: 7.9%  skip:81.0%  L0:27.3% L1:26.9% BI:45.8%
[libx264 @ 0xa68e31180] 8x8 transform intra:35.7% inter:2.2%
[libx264 @ 0xa68e31180] coded y,uvDC,uvAC intra: 69.2% 36.6% 20.5% inter: 11.3% 0.1% 0.0%
[libx264 @ 0xa68e31180] i16 v,h,dc,p: 35% 54% 12%  0%
[libx264 @ 0xa68e31180] i8 v,h,dc,ddl,ddr,vr,hd,vl,hu: 32% 17% 39%  6%  0%  0%  0%  6%  1%
[libx264 @ 0xa68e31180] i4 v,h,dc,ddl,ddr,vr,hd,vl,hu: 22% 32% 26%  2%  3%  3%  2%  3%  7%
[libx264 @ 0xa68e31180] i8c dc,h,v,p: 70% 23%  7%

[mov,mp4,m4a,3gp,3g2,mj2 @ 0x9d6420500] Auto-inserting h264_mp4toannexb bitstream filter
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x9d6420500] Auto-inserting h264_mp4toannexb bitstream filter
[mp4 @ 0x9d6421400] Starting second pass: moving the moov atom to the beginning of the file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x9d6420280] Auto-inserting h264_mp4toannexb bitstream filter
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x9d6420280] Auto-inserting h264_mp4toannexb bitstream filter
[mp4 @ 0x9d6421180] Starting second pass: moving the moov atom to the beginning of the file


{'episode_lengths': (59, 58, 57, 55, 51, 58, 49, 49, 44, 51),
 'target': '/Users/xzhu/Documents/VLA/leport/outputs/lerobot-image-demo',
 'tasks': ('lift the cube',),
 'total_episodes': 10,
 'total_frames': 531}


objc[76940]: Class AVFFrameReceiver is implemented in both /Users/xzhu/Documents/VLA/leport/.venv/lib/python3.12/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x11c3ac3a8) and /opt/homebrew/Cellar/ffmpeg/7.1_4/lib/libavdevice.61.3.100.dylib (0x135630328). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[76940]: Class AVFAudioReceiver is implemented in both /Users/xzhu/Documents/VLA/leport/.venv/lib/python3.12/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x11c3ac3f8) and /opt/homebrew/Cellar/ffmpeg/7.1_4/lib/libavdevice.61.3.100.dylib (0x135630378). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


## 6. Validate: compare source expectations with the target

With a plan, validation reinspects the source selection and checks target episodes, frames, features, and tasks.

In [8]:
validation_report = validate(target_root, plan=plan_path)
pprint(validation_report.to_dict())

{'decoded_visual_features': ['observation.images.agentview',
                             'observation.images.wrist'],
 'episode_lengths': [59, 58, 57, 55, 51, 58, 49, 49, 44, 51],
 'features': {'action': {'dtype': 'float32', 'shape': (7,)},
              'episode_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'frame_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'observation.images.agentview': {'dtype': 'video',
                                               'info': {'has_audio': False,
                                                        'is_depth_map': False,
                                                        'video.channels': 3,
                                                        'video.codec': 'h264',
                                                        'video.crf': 23,
                                                        'video.extra_options

## 7. Merge (optional): combine converted LeRobot datasets

`merge` accepts two or more **completed LeRobot dataset directories**, not raw HDF5 files. Inputs must share FPS, robot type, and the complete feature schema; tasks may differ. Episodes are renumbered in `merge_sources` order, and the merge always creates a new target directory.

The next cell skips merging by default. After preparing another compatible dataset, update `merge_sources` and set `run_merge` to `True`.

In [9]:
# This optional step is disabled so running all cells succeeds before another input is prepared.
run_merge = False

# Merge accepts converted LeRobot directories only. The first input is produced by this notebook;
# replace the second path with a dataset that has matching FPS, robot type, and feature schema.
merge_sources = (
    target_root,
    output_root / "lerobot-image-demo2",
)
merged_target_root = output_root / "lerobot-merged-demo"

if not run_merge:
    print("Optional merge skipped. Set run_merge to True after preparing another LeRobot dataset.")
else:
    # Reporting all missing inputs at once avoids repeated runs to discover paths one by one.
    missing_merge_sources = [source for source in merge_sources if not source.is_dir()]
    if missing_merge_sources:
        raise FileNotFoundError(f"LeRobot input directories do not exist: {missing_merge_sources}")
    if merged_target_root.exists():
        raise FileExistsError(
            f"Merge target already exists: {merged_target_root}. "
            "LePort does not overwrite or append datasets."
        )

    merge_result = merge(
        merge_sources,
        target_root=merged_target_root,
        repo_id="local/robomimic-merged-demo",
    )
    pprint(merge_result.to_dict())

Optional merge skipped. Set run_merge to True after preparing another LeRobot dataset.


## Equivalent CLI commands

Run CLI commands from the repository root through `uv run ...`:

```bash
uv run leport inspect data/robomimic/test/test_v15.hdf5 --adapter robomimic --json

uv run leport plan \
  --source data/robomimic/test/test_v15.hdf5 \
  --output outputs/robomimic-image-demo.yaml \
  --adapter robomimic \
  --target outputs/lerobot-image-demo \
  --repo-id local/robomimic-image-demo \
  --robot-type panda --fps 20 --task "lift the cube" \
  --action actions --action-dtype float32 \
  --state obs/robot0_eef_pos \
  --state obs/robot0_eef_quat \
  --state obs/robot0_gripper_qpos \
  --state-dtype float32 \
  --image obs/agentview_image=observation.images.agentview \
  --image obs/robot0_eye_in_hand_image=observation.images.wrist

uv run leport convert --config outputs/robomimic-image-demo.yaml --json
uv run leport validate outputs/lerobot-image-demo --config outputs/robomimic-image-demo.yaml --json

# Optional: both inputs must be completed LeRobot dataset directories with compatible schemas.
uv run leport merge \
  outputs/lerobot-image-demo \
  outputs/lerobot-image-demo2 \
  --target outputs/lerobot-merged-demo \
  --repo-id local/robomimic-merged-demo \
  --json
```

> Before rerunning conversion, review and handle existing plan and target paths explicitly. Neither the notebook nor LePort overwrites them silently.